# Day 8: Python internals: copies, iterators, dunders, MRO, caching (Exercises)

**Beyond Python for Everybody.** This is the material interviewers use to tell someone who *uses* Python apart from someone who *understands* it. Reference: the Python data model docs (docs.python.org/3/reference/datamodel.html) and *Fluent Python*.

**Why it matters at ~4 years of experience:** you will be asked *why* things behave the way they do: why a set of your objects breaks, why memory grows, why a generator is empty the second time, how `super()` really works.

**Time:** about 100 minutes · 8 exercises

**How to use this notebook**
1. Run the first cell (it defines `check()`).
2. For each exercise: read the prompt, write your code in the code cell, run it, then run the check cell below it.
3. ✅ means it passed. ❌ prints a hint. Fix and re-run until green.
4. Don't open `solutions.ipynb` until you've tried. If stuck for 10 minutes, read only the hint.
5. Run the last cell for your score, then answer the *Say it out loud* questions.

In [ ]:
# Run this cell first. It gives you the check() helper used below.
_results = {}

def check(name, test):
    """Run a test function and print a pass/fail line with a hint."""
    try:
        test()
    except AssertionError as e:
        _results[name] = False
        print(f"❌ {name}: {e}")
    except Exception as e:
        _results[name] = False
        print(f"❌ {name}: {type(e).__name__}: {e}")
    else:
        _results[name] = True
        print(f"✅ {name}")

def raises(exc, fn, *args, **kwargs):
    """True if fn(*args, **kwargs) raises exc."""
    try:
        fn(*args, **kwargs)
    except exc:
        return True
    except Exception:
        return False
    return False

TOTAL = 8

def score():
    done = sum(_results.values())
    print(f"Score: {done}/{TOTAL}  " + ("🎉 all green!" if done == TOTAL else "keep going, re-run the cells you missed"))

In [ ]:
import copy
import functools
import time
from collections import OrderedDict
from contextlib import contextmanager

## Exercise 1: Predict the output: copies and identity  `warm-up`

Read the code in the next cell **without running it** and fill in the five `answer_*` variables. Then run the check.

<details><summary>Hint</summary>

`copy.copy` makes a new outer list but shares the inner lists. `deepcopy` copies everything. A tuple can't be reassigned, but a list *inside* a tuple can still change.

</details>

In [ ]:
a = [[1, 2], [3]]
b = copy.copy(a)
c = copy.deepcopy(a)
a[0].append(99)

t = (1, [2])
t[1].append(3)

answer_b0 = None           # what is b[0] now?
answer_c0 = None           # what is c[0] now?
answer_b_is_a = None       # True or False: b is a
answer_b0_is_a0 = None     # True or False: b[0] is a[0]
answer_t = None            # what is t now?
answer_t_assign = None     # which exception does `t[0] = 5` raise? (a string like "KeyError")

In [ ]:
def _test():
    assert answer_b0 == [1, 2, 99], "a shallow copy shares the inner lists, so b[0] sees the append"
    assert answer_c0 == [1, 2], "deepcopy copied the inner lists too"
    assert answer_b_is_a is False, "copy.copy builds a new outer list"
    assert answer_b0_is_a0 is True, "the inner list is the same object in both"
    assert answer_t == (1, [2, 3]), "the tuple can't change which objects it holds, but the list inside can change"
    assert answer_t_assign == "TypeError", "tuples don't support item assignment"

check('1. Predict the output: copies and identity', _test)

## Exercise 2: Countdown: iterable vs iterator  `medium`

Write two classes:
- `CountdownIterator(start)`: an **iterator**. `__next__` returns `start`, `start - 1`, ... down to `1`, then raises `StopIteration`. `__iter__` returns `self`.
- `Countdown(start)`: an **iterable**. Its `__iter__` returns a **new** `CountdownIterator` each time.

`list(Countdown(3))` -> `[3, 2, 1]`, and looping over the same `Countdown` twice gives the numbers both times. An iterator, by contrast, is used up after one pass.

<details><summary>Hint</summary>

This split is why you can loop over a list many times but a generator only once: a list is an iterable that makes a fresh iterator each time.

</details>

In [ ]:
class CountdownIterator:
    pass


class Countdown:
    pass

In [ ]:
def _test():
    assert list(Countdown(3)) == [3, 2, 1], "Countdown(3) should give 3, 2, 1"
    c = Countdown(2)
    assert list(c) == [2, 1] and list(c) == [2, 1], "an iterable must work twice: return a NEW iterator from __iter__"
    it = iter(Countdown(1))
    assert iter(it) is it, "an iterator's __iter__ returns self"
    assert next(it) == 1
    assert raises(StopIteration, next, it), "after the last value, __next__ must raise StopIteration"
    assert list(Countdown(0)) == []

check('2. Countdown: iterable vs iterator', _test)

## Exercise 3: Vector: dunder methods and hashing  `medium`

Write `Vector(x, y)` with:
- `__repr__` -> `"Vector(1, 2)"`
- `__eq__`: equal when `x` and `y` match (return `NotImplemented` for non-Vectors)
- `__hash__`: consistent with `__eq__`, so equal vectors work as one set member or dict key
- `__add__` -> a new `Vector`
- `__len__` -> `2`, and `__getitem__(i)` -> `x` for `0`, `y` for `1`, `IndexError` otherwise
- `__bool__` -> `False` only for `Vector(0, 0)`

Interview point: if you define `__eq__` but not `__hash__`, Python sets `__hash__` to `None` and the class can't go in a set.

<details><summary>Hint</summary>

`hash((self.x, self.y))` is the usual trick. Equal objects **must** have equal hashes. Hash only things that won't change after creation.

</details>

In [ ]:
class Vector:
    pass

In [ ]:
def _test():
    v = Vector(1, 2)
    assert repr(v) == "Vector(1, 2)", "repr should look like Vector(1, 2)"
    assert v == Vector(1, 2) and v != Vector(2, 1), "compare by x and y"
    assert len({Vector(1, 2), Vector(1, 2)}) == 1, "equal vectors must hash the same (define __hash__)"
    assert {Vector(1, 2): "a"}[Vector(1, 2)] == "a", "an equal vector should find the dict key"
    assert v + Vector(3, 4) == Vector(4, 6)
    assert len(v) == 2 and v[0] == 1 and v[1] == 2
    assert raises(IndexError, lambda: v[2]), "v[2] should raise IndexError"
    assert list(v) == [1, 2], "with __getitem__ raising IndexError, list(v) works for free"
    assert not Vector(0, 0) and Vector(0, 3), "only Vector(0, 0) is falsy"
    assert (v == "nope") is False

check('3. Vector: dunder methods and hashing', _test)

## Exercise 4: Predict the output: MRO and super() in a diamond  `interview`

Read the classes **without running them**. When `Child(log)` is created, in what order are names appended to `log`? And what is `Child.__mro__` (as a list of class names)?

Interview point: `super()` doesn't mean "my parent". It means "the next class in the MRO of the object's actual class".

<details><summary>Hint</summary>

Python uses C3 linearization: the child first, then parents left to right, and a shared base comes only after all the classes that inherit from it. `Base` runs exactly once.

</details>

In [ ]:
class Base:
    def __init__(self, log):
        log.append("Base")

class Left(Base):
    def __init__(self, log):
        log.append("Left")
        super().__init__(log)

class Right(Base):
    def __init__(self, log):
        log.append("Right")
        super().__init__(log)

class Child(Left, Right):
    def __init__(self, log):
        log.append("Child")
        super().__init__(log)

predicted_log = None   # e.g. ["Child", ...]
predicted_mro = None   # e.g. ["Child", ..., "object"]

In [ ]:
def _test():
    log = []
    Child(log)
    assert predicted_mro == [c.__name__ for c in Child.__mro__], "check the MRO: child, then parents left to right, shared base last"
    assert predicted_log == log, "super() follows the MRO of Child, so Left's super() is Right, not Base"

check('4. Predict the output: MRO and super() in a diamond', _test)

## Exercise 5: @property validation and __slots__  `medium`

1. `Temperature(celsius)`: `celsius` is a property whose setter raises `ValueError` below `-273.15` (also when set in `__init__`). `fahrenheit` is a **read-only** property: `celsius * 9 / 5 + 32`.
2. `Point(x, y)` using `__slots__ = ("x", "y")`. Setting any other attribute must raise `AttributeError`.

Interview point: `__slots__` removes the per-instance `__dict__`, which saves memory when you have millions of small objects.

<details><summary>Hint</summary>

In `__init__`, write `self.celsius = celsius` (not `self._celsius`) so the setter's validation runs. A property with no setter is read-only.

</details>

In [ ]:
class Temperature:
    pass


class Point:
    pass

In [ ]:
def _test():
    t = Temperature(100)
    assert t.fahrenheit == 212, "100°C is 212°F"
    t.celsius = 0
    assert t.fahrenheit == 32, "fahrenheit must be computed from the current celsius"
    assert raises(ValueError, setattr, t, "celsius", -300), "the setter must reject values below -273.15"
    assert raises(ValueError, Temperature, -500), "validate in __init__ too: assign through the property"
    assert raises(AttributeError, setattr, t, "fahrenheit", 50), "fahrenheit should be read-only (no setter)"
    p = Point(1, 2)
    assert (p.x, p.y) == (1, 2)
    assert raises(AttributeError, setattr, p, "z", 3), "with __slots__, new attributes are not allowed"
    assert not hasattr(p, "__dict__"), "a slotted instance has no __dict__"

check('5. @property validation and __slots__', _test)

## Exercise 6: temp_setting: a context manager with contextlib  `medium`

Write `temp_setting(config, key, value)` with `@contextmanager`. Inside the `with` block, `config[key] == value`. On exit, even if the block raised, put things back:
- if the key existed, restore its old value
- if it didn't exist, remove it

The exception must still propagate (don't swallow it). This pattern is used for feature flags, test overrides and `unittest.mock.patch.dict`.

<details><summary>Hint</summary>

Put `yield` inside `try:` and the restore in `finally:`. Use a sentinel like `missing = object()` to tell "the key was absent" apart from "the value was None".

</details>

In [ ]:
def temp_setting(config, key, value):
    pass

In [ ]:
def _test():
    cfg = {"debug": False}
    with temp_setting(cfg, "debug", True):
        assert cfg["debug"] is True, "inside the block the new value applies"
    assert cfg == {"debug": False}, "the old value must be restored"
    with temp_setting(cfg, "region", "eu"):
        assert cfg["region"] == "eu"
    assert "region" not in cfg, "a key that didn't exist must be removed again"
    try:
        with temp_setting(cfg, "debug", True):
            raise ValueError("boom")
    except ValueError:
        pass
    else:
        raise AssertionError("don't swallow the exception from the block")
    assert cfg == {"debug": False}, "restore even when the block raises (use try/finally)"

check('6. temp_setting: a context manager with contextlib', _test)

## Exercise 7: functools.wraps and lru_cache  `medium`

1. Write a decorator `timed` that stores how long the last call took (in seconds) on `wrapper.last_elapsed`, and uses `functools.wraps` so the decorated function keeps its `__name__` and `__doc__`.
2. Add `@functools.lru_cache(maxsize=128)` to `get_rate` so repeated currencies don't do the slow lookup again (the global `RATE_LOOKUPS` counts real lookups).

Interview point: without `wraps`, every decorated function would be called `wrapper` in logs, tracebacks and FastAPI docs.

<details><summary>Hint</summary>

Set `wrapper.last_elapsed` in a `finally:` so it's recorded even if the function raises. `get_rate.cache_info()` shows hits and misses, and `get_rate.cache_clear()` empties the cache.

</details>

In [ ]:
def timed(func):
    pass


RATE_LOOKUPS = 0

def get_rate(currency):
    """Pretend this is a slow API call."""
    global RATE_LOOKUPS
    RATE_LOOKUPS += 1
    return {"USD": 1.0, "EUR": 0.92, "INR": 83.0}[currency]

In [ ]:
def _test():
    @timed
    def add(a, b):
        """Add two numbers."""
        return a + b
    assert add(1, 2) == 3, "the wrapper must return the function's result"
    assert add.__name__ == "add" and add.__doc__ == "Add two numbers.", "use @functools.wraps(func) on the wrapper"
    assert isinstance(add.last_elapsed, float) and add.last_elapsed >= 0, "store the elapsed seconds on wrapper.last_elapsed"

    assert hasattr(get_rate, "cache_info"), "decorate get_rate with @functools.lru_cache(maxsize=128)"
    get_rate.cache_clear()
    globals()["RATE_LOOKUPS"] = 0
    for cur in ["USD", "USD", "EUR", "USD"]:
        get_rate(cur)
    assert RATE_LOOKUPS == 2, "only USD and EUR should hit the slow lookup"
    assert get_rate.cache_info().hits == 2

check('7. functools.wraps and lru_cache', _test)

## Exercise 8: Interview: LRUCache with OrderedDict  `interview`

Implement `LRUCache(capacity)`:
- `get(key)` returns the value, or `-1` if missing. A hit makes the key the most recently used.
- `put(key, value)` inserts or updates. If the cache is over capacity, evict the **least** recently used key.

Both must be O(1). This is a very common live-coding question, and the follow-up is usually "how would you do it without OrderedDict?" (a dict plus a doubly linked list).

<details><summary>Hint</summary>

`OrderedDict.move_to_end(key)` marks a key as most recent. `popitem(last=False)` removes the oldest.

</details>

In [ ]:
class LRUCache:
    pass

In [ ]:
def _test():
    c = LRUCache(2)
    c.put(1, "one")
    c.put(2, "two")
    assert c.get(1) == "one"
    c.put(3, "three")            # evicts 2, because 1 was used more recently
    assert c.get(2) == -1, "2 was the least recently used, so it should be evicted"
    c.put(4, "four")             # evicts 1
    assert c.get(1) == -1, "after get(1), then put(3), the oldest is 1 when 4 arrives"
    assert c.get(3) == "three" and c.get(4) == "four"
    c.put(3, "THREE")            # update moves 3 to most recent
    c.put(5, "five")             # evicts 4
    assert c.get(4) == -1 and c.get(3) == "THREE", "updating a key also makes it most recent"

check('8. Interview: LRUCache with OrderedDict', _test)

## Your score

In [ ]:
score()

## Say it out loud (interview questions)

Answer each one aloud in under 60 seconds, as if a client asked. Model answers are in `solutions.ipynb`.

- A client's CPU-heavy Python script doesn't get any faster with 8 threads. What do you tell them?
- A long-running service's memory keeps growing even though you `del` objects. How does Python free memory, and what could be keeping objects alive?
- You added `__eq__` to a model class and now `set(users)` raises `TypeError: unhashable type`. Why, and how do you fix it safely?
- In a diamond inheritance, a teammate is surprised that calling `super().__init__()` in `Left` ran `Right.__init__`. Explain.
- A function works when you pass a list, but processes nothing the second time you pass it a generator. Why?